<a href="https://colab.research.google.com/github/UdaraChamidu/Eye-Disease-Classification-With-Integrated-Chatbot/blob/main/Fine_Tune_bioGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install Required Libraries

In [7]:
pip install transformers datasets accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

# Step 2: Load Dataset (.jsonl)

In [8]:
from datasets import Dataset
import json

# Load your JSONL file
file_path = "/content/cleaned_dataset.jsonl"

# Step 1: Read the lines into a list of dicts
with open(file_path, "r", encoding="utf-8") as f:
    data_lines = [json.loads(line) for line in f]

# Step 2: Convert to Hugging Face Dataset
dataset = Dataset.from_list(data_lines)

# Step 3: Format the dataset correctly using the right keys
def format_example(example):
    return {
        "text": f"### Question:\n{example['prompt']}\n\n### Answer:\n{example['completion']}"
    }

# Step 4: Apply formatting
dataset = dataset.map(format_example)


Map:   0%|          | 0/16591 [00:00<?, ? examples/s]

In [9]:
print(dataset[0]["text"])

### Question:
Given your profession as an ophthalmologist, please provide a detailed and comprehensive response to the Question:.For the past week or so I have been excessively rubbing my left eye for some reason. I think it’s a nervous habit. I don’t know how many times I’ve done it but it’s quite a bit. But I am now worried that the rubbing I have done is enough to cause keratoconus (a cornea that becomes thin and misshapened over time). Is it possible that I can get this  the rubbing I have done?


### Answer:
Eye rubbing is a risk factor for development of keratoconus. Certain individuals (for example younger people and those with connective tissue disorders such as Ehlers-Danlos syndrome and osteogenesis imperfecta) are more susceptible to the effects of eye rubbing. You should see your ophthalmologist to see if you are at risk for keratoconus and if the eye rubbing is due to an underlying eye disease such as atopic or allergic conjunctivitis (swelling and inflammation of the whit

# Step 3: Load the Pretrained BioGPT Model

In [10]:
!pip install -q sacremoses


In [11]:
from transformers import BioGptTokenizer, BioGptForCausalLM

tokenizer = BioGptTokenizer.from_pretrained("microsoft/biogpt")
model = BioGptForCausalLM.from_pretrained("microsoft/biogpt")
# model_name = "microsoft/biogpt"

# Step 4: Tokenize the Data

In [12]:
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=512)

tokenized_data = dataset.map(tokenize, batched=True, remove_columns=['prompt', 'completion', 'text'])


Map:   0%|          | 0/16591 [00:00<?, ? examples/s]

# Step 5: Define Training Arguments

In [13]:
!pip uninstall -y transformers
!pip install transformers --upgrade


Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)


In [14]:
import transformers
print(transformers.__version__)


4.52.4


In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./biogpt-finetuned",
    #evaluation_strategy="no",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)


# Step 6: Initialize Trainer and Train

In [16]:
from transformers import BioGptTokenizer

tokenizer = BioGptTokenizer.from_pretrained("microsoft/biogpt")


In [17]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()


/tmp/ipython-input-17-4157699553.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
10,2.729300
20,2.408800
30,2.104100
40,2.264500
50,2.371500
60,2.130500
70,2.163500
80,2.543900
90,2.314900
100,2.059900


TrainOutput(global_step=12444, training_loss=1.6097110331307574, metrics={'train_runtime': 7695.7089, 'train_samples_per_second': 6.468, 'train_steps_per_second': 1.617, 'total_flos': 4.622421966874214e+16, 'train_loss': 1.6097110331307574, 'epoch': 3.0})

# Step 7: Save and Use Fine-Tuned Model

In [18]:
trainer.save_model("./biogpt-finetuned")
tokenizer.save_pretrained("./biogpt-finetuned")


('./biogpt-finetuned/tokenizer_config.json',
 './biogpt-finetuned/special_tokens_map.json',
 './biogpt-finetuned/vocab.json',
 './biogpt-finetuned/merges.txt',
 './biogpt-finetuned/added_tokens.json')

In [19]:
from transformers import pipeline

pipe = pipeline("text-generation", model="./biogpt-finetuned", tokenizer="./biogpt-finetuned")
pipe("### Question:\nWhat is cataract?\n\n### Answer:\n", max_new_tokens=100)


Device set to use cuda:0


[{'generated_text': '### Question:\nWhat is cataract?\n\n### Answer:\n Cataract is a clouding of the lens in the eye. Cataracts are a major cause of blindness in the United States and worldwide. The most common types of cataract are senile (age-related) cataracts, which are the most common form of cataract. As with any clouding of the eye, cataract surgery is required to remove the lens and replace it with an artificial lens. It is important to have a clear and intact anterior capsule in the surgery. If you have a cloudy lens in'}]

# Download the Fine-tuned model

In [20]:
!zip -r biogpt-finetuned.zip ./biogpt-finetuned

  adding: biogpt-finetuned/ (stored 0%)
  adding: biogpt-finetuned/training_args.bin (deflated 51%)
  adding: biogpt-finetuned/tokenizer_config.json (deflated 73%)
  adding: biogpt-finetuned/config.json (deflated 49%)
  adding: biogpt-finetuned/model.safetensors (deflated 7%)
  adding: biogpt-finetuned/special_tokens_map.json (deflated 51%)
  adding: biogpt-finetuned/checkpoint-12444/ (stored 0%)
  adding: biogpt-finetuned/checkpoint-12444/training_args.bin (deflated 51%)
  adding: biogpt-finetuned/checkpoint-12444/tokenizer_config.json (deflated 73%)
  adding: biogpt-finetuned/checkpoint-12444/config.json (deflated 49%)
  adding: biogpt-finetuned/checkpoint-12444/model.safetensors (deflated 7%)
  adding: biogpt-finetuned/checkpoint-12444/trainer_state.json (deflated 81%)
  adding: biogpt-finetuned/checkpoint-12444/special_tokens_map.json (deflated 51%)
  adding: biogpt-finetuned/checkpoint-12444/optimizer.pt (deflated 10%)
  adding: biogpt-finetuned/checkpoint-12444/scheduler.pt (defl

In [21]:
from google.colab import files
files.download("biogpt-finetuned.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Use the downloaded model anywhere

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "path_to/biogpt-finetuned"  # unzip if needed

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Push to hugging Face

In [ ]:
!pip install -q huggingface_hub


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
repo_name = "biogpt-finetuned-eye-disease"  # You can change this


In [ ]:
from huggingface_hub import HfApi, create_repo, upload_folder
from transformers import AutoTokenizer, AutoModelForCausalLM

# Push directory
model_dir = "./biogpt-finetuned"

# Create repo (if not exists)
create_repo(repo_name, exist_ok=True)

# Upload the whole folder
upload_folder(
    folder_path=model_dir,
    repo_id=f"{your_username}/{repo_name}",  # Replace with your Hugging Face username
    commit_message="Upload fine-tuned BioGPT model"
)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)


# Load my model from Hugging Face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("your_username/biogpt-finetuned-eye-disease")
model = AutoModelForCausalLM.from_pretrained("your_username/biogpt-finetuned-eye-disease")
